# Goodfire nano parameter decomposition on CPU

Install `.[goodfire-local,notebooks]`. Runs the pinned upstream nano VPD implementation on a tiny local token model for two steps, with reconstruction, importance and stochastic/adversarial losses. This is an execution check, NOT a converged decomposition. The upstream nano L_p objective differs from the current JAX smooth-L0 trainer. W&B logging is disabled; no hosted Goodfire API is used.

Run all cells in a fresh kernel. The final cell records local runtime and kernel RSS.

In [1]:
import os, time, json, resource, sys
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["USE_TF"] = "0"
os.environ["WANDB_MODE"] = "disabled"
import torch, numpy as np
torch.set_num_threads(2)
torch.set_num_interop_threads(2)
torch.manual_seed(7)
np.random.seed(7)
started = time.perf_counter()
print("CPU, fixed seed 7; tiny synthetic fixtures; no weight downloads")


CPU, fixed seed 7; tiny synthetic fixtures; no weight downloads


In [2]:
import itertools
from torch import nn
from autoexplain.frontier import run_goodfire_nano
model = nn.Sequential(nn.Embedding(16,4), nn.Linear(4,16))
batch = torch.tensor([[1,2,3,4], [2,3,4,5]])
with torch.no_grad(): original = model(batch).clone()
stream = itertools.cycle([batch])
model = run_goodfire_nano(model, {"1": 4}, stream, stream, steps=2)
with torch.no_grad():
    print("Target-mode preservation error:", (model(batch)-original).abs().max().item())
    wrapper = model[1]
    wrapper.mode = "component"
    wrapper.mask = torch.ones(2,4,4)
    wrapper.delta_mask = torch.zeros(2,4)
    reconstructed = model(batch)
    wrapper.mask[...,0] = 0
    ablated = model(batch)
    wrapper.mode = "target"; wrapper.mask = None; wrapper.delta_mask = None
print("Component ablation change:", (ablated-reconstructed).norm().item())
assert torch.isfinite(reconstructed).all()
print("Two steps cannot establish semantic decomposition quality.")


[rank0] installed 1 components


[rank0] built CI transformer (612 params)


[rank0] moved models to cpu


[rank0] starting faithfulness warmup (2 steps)


[rank0] faithfulness warmup done


[rank0] DDP wrap complete


[rank0] PPGD, optimizer, dataloader ready


loss/faith=0.2798 loss/imp=0.3638 loss/stoch=0.06189 loss/ppgd=0.0641 lr/main=5e-05 lr/ppgd=0.01 step=0


loss/faith=0.2797 loss/imp=0.7787 loss/stoch=0.05888 loss/ppgd=0.07002 lr/main=5e-06 lr/ppgd=0.01 step=1


Target-mode preservation error: 0.0
Component ablation change: 4.452613353729248
Two steps cannot establish semantic decomposition quality.


In [3]:
elapsed = time.perf_counter() - started
rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
rss_mib = rss / (1024**2 if sys.platform == "darwin" else 1024)
print(json.dumps({"tutorial_compute_wall_seconds": round(elapsed, 3),
                 "kernel_lifetime_peak_rss_mib": round(rss_mib, 2),
                 "memory_scope": "kernel only, including imports; not aggregate system memory"}))


{"tutorial_compute_wall_seconds": 1.41, "kernel_lifetime_peak_rss_mib": 446.02, "memory_scope": "kernel only, including imports; not aggregate system memory"}
